In [1]:
# =========================
# 1) Mount Google Drive
# =========================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls /content/drive/MyDrive/yhd_project/llm-main

LLMjudge_eval_on_summeval_updated.ipynb  summeval_gold_flat.jsonl
llmjudge_eval_summeval.py		 summeval_sample500_seed42.jsonl
prompt.txt


In [3]:
# =========================
# 1) Install
# Restart runtime after this cell if Colab asks.
# =========================
!pip -q install -U transformers datasets trl peft accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 520.7/520.7 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.6 MB/s eta 0:00:00


In [4]:

!pip install -q datasets
from datasets import load_dataset

ds = load_dataset("mteb/summeval")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/test-00000-of-00001-35901af5f664939(…):   0%|          | 0.00/423k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

In [5]:
###predicting on summeval

In [6]:
from pathlib import Path
import os

print("cwd:", os.getcwd())
print("content exists:", Path("/content").exists())
print("drive root exists:", Path("/content/drive").exists())
print("mydrive exists:", Path("/content/drive/MyDrive").exists())
print("llm_judge exists:", Path("/content/drive/MyDrive/yhd_project/llm-main").exists())
print("pred file exists:", Path("/content/drive/MyDrive/yhd_project/llm-main/qwen25_1.5b_summeval_preds.jsonl").exists())

cwd: /content
content exists: True
drive root exists: True
mydrive exists: True
llm_judge exists: True
pred file exists: False


In [7]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [8]:
#!/usr/bin/env python
"""Run inference with Qwen2.5-1B LoRA judge and save predictions to JSONL."""

import argparse
import json
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
LIMIT = 0
max_new_tokens = 320
DEFAULT_BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
DEFAULT_ADAPTER_PATH = Path("/content/drive/MyDrive/yhd_project/final_weights")  # make sure this is the 3B adapter
DEFAULT_INPUT_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/summeval_sample500_seed42.jsonl")
DEFAULT_OUTPUT_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/qwen25_3b_summeval_preds.jsonl")
PROMPT_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/prompt.txt")


VAL_RATIO = 0.1
SEED = 42
REQUIRED_KEYS = ("coherence", "consistency", "fluency", "relevance")
def build_input(base_prompt: str, article: str, summary: str) -> str:
    return (
        f"{base_prompt}\n\n"
        "This is the complete source article for which you will be evaluating:\n"
        f"{article}\n\n"
        "This is the LLM generated summary you are about to evaluate:\n"
        f"{summary}\n\n"
        "Return ONLY the final JSON object."
    )


def load_rows(path: Path) -> list[dict]:
    if path.suffix.lower() == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8") as fin:
            for line in fin:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows
    return json.loads(path.read_text(encoding="utf-8"))


def extract_json(raw: str) -> dict | None:
    raw = (raw or "").strip()
    if not raw:
        return None
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1 or end <= start:
        return None
    candidate = raw[start : end + 1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return None


def get_prompt_for_row(row: dict, base_prompt: str) -> str:
    if isinstance(row.get("messages"), list) and row["messages"]:
        for m in row["messages"]:
            if m.get("role") == "user":
                return m.get("content", "")

    article = row.get("original", "")
    summary = row.get("summary", "")
    if article and summary:
        return build_input(base_prompt, article, summary)

    raise ValueError("Row missing both `messages` and `original`+`summary`.")


def extract_article_summary_from_prompt(prompt: str) -> tuple[str | None, str | None]:
    a_marker = "This is the complete source article for which you will be evaluating:\n"
    s_marker = "\n\nThis is the LLM generated summary you are about to evaluate:\n"
    end_marker = "\n\nReturn ONLY the final JSON object."
    a_start = prompt.find(a_marker)
    s_start = prompt.find(s_marker)
    if a_start == -1 or s_start == -1:
        return None, None
    article = prompt[a_start + len(a_marker) : s_start]
    rem = prompt[s_start + len(s_marker) :]
    end_pos = rem.rfind(end_marker)
    summary = rem[:end_pos] if end_pos != -1 else rem
    return article.strip() or None, summary.strip() or None



base_prompt = PROMPT_PATH.read_text(encoding="utf-8")

input_path = Path(DEFAULT_INPUT_PATH)
output_path = Path(DEFAULT_OUTPUT_PATH)
adapter_path = Path(DEFAULT_ADAPTER_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

rows = load_rows(input_path)


dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
tokenizer = AutoTokenizer.from_pretrained(DEFAULT_BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_BASE_MODEL,
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
)
model = PeftModel.from_pretrained(base_model, DEFAULT_ADAPTER_PATH)
model.eval()

total = len(rows)
with output_path.open("w", encoding="utf-8") as fout:
    for i, row in enumerate(rows, start=1):
        prompt = get_prompt_for_row(row, base_prompt)
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_tensors="pt",
            return_dict=True,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[-1] :]
        pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        pred_json = extract_json(pred_text)

        out = {
            "id": row["id"],
            "article_id": row.get("article_id"),
            "summary_index": row.get("summary_index"),
            "original": row.get("original"),
            "summary": row.get("summary"),
            "gold_relevance": row.get("gold_relevance"),
            "gold_coherence": row.get("gold_coherence"),
            "gold_fluency": row.get("gold_fluency"),
            "gold_consistency": row.get("gold_consistency"),
            "prediction_text": pred_text,
            "prediction_json": pred_json,
            "parse_ok": pred_json is not None,
        }

        fout.write(json.dumps(out, ensure_ascii=False) + "\n")

        if i % 25 == 0 or i == len(rows):
            print(f"[{i}/{len(rows)}] done")
print("saved to", DEFAULT_OUTPUT_PATH)



config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[25/500] done
[50/500] done
[75/500] done
[100/500] done
[125/500] done
[150/500] done
[175/500] done
[200/500] done
[225/500] done
[250/500] done
[275/500] done
[300/500] done
[325/500] done
[350/500] done
[375/500] done
[400/500] done
[425/500] done
[450/500] done
[475/500] done
[500/500] done
saved to /content/drive/MyDrive/yhd_project/llm-main/qwen25_3b_summeval_preds.jsonl


In [9]:
!ls -lh /content/drive/MyDrive/yhd_project/llm-main

total 7.9M
-rw------- 1 root root 187K Mar  8 08:55 'Copy of LLMjudge_eval_on_summeval_updated.ipynb'
-rw------- 1 root root 153K Mar  8 11:28  LLMjudge_eval_on_summeval_updated.ipynb
-rw------- 1 root root  17K Mar  8 08:01  llmjudge_eval_summeval.py
-rw------- 1 root root 4.2K Mar  8 08:01  prompt.txt
-rw------- 1 root root 2.0M Mar  8 11:37  qwen25_3b_summeval_preds.jsonl
-rw------- 1 root root 4.2M Mar  8 08:01  summeval_gold_flat.jsonl
-rw------- 1 root root 1.4M Mar  8 08:01  summeval_sample500_seed42.jsonl


In [12]:
###evaluation table for summeval on 1.5B



import json
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr, pearsonr, kendalltau

PRED_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/qwen25_3b_summeval_preds.jsonl")
GOLD_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/summeval_gold_flat.jsonl")
DIMENSIONS = ["coherence", "consistency", "fluency", "relevance"]


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def try_parse_json(x):
    if x is None:
        return None
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        try:
            return json.loads(x)
        except:
            start = x.find("{")
            end = x.rfind("}")
            if start != -1 and end != -1 and end > start:
                try:
                    return json.loads(x[start:end+1])
                except:
                    return None
    return None


def extract_scores(obj):
    obj = try_parse_json(obj)
    out = {}
    for dim in DIMENSIONS:
        score = None
        if isinstance(obj, dict) and isinstance(obj.get(dim), dict):
            score = obj[dim].get("score")
        out[dim] = score
    return out


def build_gold_record(row):
    rec = {"id": str(row.get("id"))}
    has_flat_gold = any(row.get(f"gold_{dim}") is not None for dim in DIMENSIONS)

    if has_flat_gold:
        for dim in DIMENSIONS:
            rec[f"gold_{dim}"] = row.get(f"gold_{dim}")
        return rec

    scores = extract_scores(row.get("raw_response_text"))
    for dim in DIMENSIONS:
        rec[f"gold_{dim}"] = scores[dim]
    return rec

In [13]:
pred_rows = load_jsonl(PRED_PATH)

pred_data = []
gold_data = []
for row in pred_rows:
    pred_json = row.get("prediction_json")
    scores = extract_scores(pred_json)
    rec = {"id": str(row.get("id"))}
    for dim in DIMENSIONS:
        rec[f"pred_{dim}"] = scores[dim]
    pred_data.append(rec)

    if any(row.get(f"gold_{dim}") is not None for dim in DIMENSIONS):
        gold_data.append(build_gold_record(row))

if gold_data:
    print("Using embedded gold labels from prediction file.")
else:
    gold_rows = load_jsonl(GOLD_PATH)
    gold_data = [build_gold_record(row) for row in gold_rows]
    print(f"Using external gold file: {GOLD_PATH}")

pred_df = pd.DataFrame(pred_data)
gold_df = pd.DataFrame(gold_data)

merged = pred_df.merge(gold_df, on="id", how="inner")

for dim in DIMENSIONS:
    merged[f"pred_{dim}"] = pd.to_numeric(merged[f"pred_{dim}"], errors="coerce")
    merged[f"gold_{dim}"] = pd.to_numeric(merged[f"gold_{dim}"], errors="coerce")

usable_counts = {}
for dim in DIMENSIONS:
    mask = ~(pd.isna(merged[f"pred_{dim}"]) | pd.isna(merged[f"gold_{dim}"]))
    usable_counts[dim] = int(mask.sum())

print("pred rows:", len(pred_df))
print("gold rows:", len(gold_df))
print("matched rows:", len(merged))
print("usable rows by dimension:", usable_counts)

merged.head()

Using embedded gold labels from prediction file.
pred rows: 500
gold rows: 500
matched rows: 500
usable rows by dimension: {'coherence': 496, 'consistency': 496, 'fluency': 496, 'relevance': 496}


,id,pred_coherence,pred_consistency,pred_fluency,pred_relevance,gold_coherence,gold_consistency,gold_fluency,gold_relevance
0,cnn-test-404f859482d47c127868964a9a39d1a7645dd...,4.0,5.0,4.0,3.0,4.666667,5.000000,5.0,4.333333
1,cnn-test-404f859482d47c127868964a9a39d1a7645dd...,4.0,5.0,4.0,3.0,4.666667,5.000000,5.0,4.333333
2,cnn-test-404f859482d47c127868964a9a39d1a7645dd...,4.0,5.0,4.0,3.0,2.666667,5.000000,5.0,1.666667
3,cnn-test-404f859482d47c127868964a9a39d1a7645dd...,4.0,5.0,4.0,3.0,4.666667,5.000000,5.0,4.666667
4,cnn-test-404f859482d47c127868964a9a39d1a7645dd...,2.0,2.0,2.0,2.0,2.000000,1.333333,2.0,1.666667


In [14]:
def safe_corr(fn, x, y):
    mask = ~(pd.isna(x) | pd.isna(y))
    x = np.array(x[mask], dtype=float)
    y = np.array(y[mask], dtype=float)
    if len(x) < 2:
        return np.nan
    try:
        val = fn(x, y)[0]
        return val
    except:
        return np.nan


def evaluate_dimension(df, dim):
    pred = df[f"pred_{dim}"]
    gold = df[f"gold_{dim}"]

    mask = ~(pd.isna(pred) | pd.isna(gold))
    pred = np.array(pred[mask], dtype=float)
    gold = np.array(gold[mask], dtype=float)

    if len(pred) == 0:
        return {
            "dimension": dim,
            "n": 0,
            "spearman": np.nan,
            "pearson": np.nan,
            "kendall": np.nan,
            "mae": np.nan,
            "rmse": np.nan,
            "exact_match": np.nan,
            "off_by_1_acc": np.nan,
            "pred_mean": np.nan,
            "gold_mean": np.nan,
        }

    return {
        "dimension": dim,
        "n": len(pred),
        "spearman": safe_corr(spearmanr, pred, gold),
        "pearson": safe_corr(pearsonr, pred, gold),
        "kendall": safe_corr(kendalltau, pred, gold),
        "mae": np.mean(np.abs(pred - gold)),
        "rmse": np.sqrt(np.mean((pred - gold) ** 2)),
        "exact_match": np.mean(pred == gold),
        "off_by_1_acc": np.mean(np.abs(pred - gold) <= 1),
        "pred_mean": np.mean(pred),
        "gold_mean": np.mean(gold),
    }


eval_rows = [evaluate_dimension(merged, dim) for dim in DIMENSIONS]
eval_df = pd.DataFrame(eval_rows)

eval_df_rounded = eval_df.copy()

for col in ["spearman", "pearson", "kendall", "mae", "rmse", "exact_match", "off_by_1_acc", "pred_mean", "gold_mean"]:
    eval_df_rounded[col] = eval_df_rounded[col].round(4)

eval_df_rounded



,dimension,n,spearman,pearson,kendall,mae,rmse,exact_match,off_by_1_acc,pred_mean,gold_mean
0,coherence,496,0.4821,0.5040,0.4036,0.7446,0.9457,0.1472,0.7742,3.6956,3.4603
1,consistency,496,0.6108,0.7026,0.5801,0.2608,0.7030,0.7802,0.9315,4.6794,4.6391
2,fluency,496,0.4278,0.4535,0.3973,1.1492,1.2995,0.0847,0.6633,3.5907,4.6727
3,relevance,496,0.3150,0.3901,0.2686,0.9079,1.0881,0.1069,0.6512,3.0262,3.7849


In [15]:
OUT_DIR = Path("/content/drive/MyDrive/yhd_project/llm-main/summ_eval")
OUT_DIR.mkdir(parents=True, exist_ok=True)
eval_df_rounded.to_csv(OUT_DIR / "eval_tables_summeval_qwen3B.csv", index=False)


In [ ]:
# import pandas as pd
# from datasets import load_dataset
# import json
# from pathlib import Path
# import pandas as pd

# ds = load_dataset("mteb/summeval")
# rows = []

# for ex in ds["test"]:
#     article_id = ex["id"]
#     text = ex["text"]

#     summaries = ex["machine_summaries"]
#     rels = ex["relevance"]
#     cohs = ex["coherence"]
#     flus = ex["fluency"]
#     cons = ex["consistency"]

#     n = len(summaries)

#     assert len(rels) == n
#     assert len(cohs) == n
#     assert len(flus) == n
#     assert len(cons) == n

#     for i in range(n):
#         rows.append({
#             "id": f"{article_id}__sys{i}",
#             "article_id": article_id,
#             "summary_index": i,
#             "original": text,
#             "summary": summaries[i],
#             "gold_relevance": rels[i],
#             "gold_coherence": cohs[i],
#             "gold_fluency": flus[i],
#             "gold_consistency": cons[i],
#         })

# gold_df = pd.DataFrame(rows)
# out_path = Path("/content/drive/MyDrive/yhd_project/llm-main/summeval_gold_flat.jsonl")

# with open(out_path, "w", encoding="utf-8") as f:
#     for row in rows:
#         f.write(json.dumps(row, ensure_ascii=False) + "\n")

# print("saved:", out_path)
# print("rows:", len(rows))

Cell below here is for gpt5 data no need to run this

In [ ]:
# from pathlib import Path

In [ ]:
# #!/usr/bin/env python
# """Run inference with Qwen2.5-3B LoRA judge and save predictions to JSONL."""

# import argparse
# import json
# from pathlib import Path

# import torch
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
# LIMIT = 0
# max_new_tokens = 320
# DEFAULT_BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# DEFAULT_ADAPTER_PATH = Path("/content/drive/MyDrive/yhd_project/final_weights")
# DEFAULT_INPUT_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/sft_val.jsonl")
# DEFAULT_OUTPUT_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/sft_val_qwen25_1.5b_lora_test_preds_gpt5.jsonl")
# PROMPT_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/prompt.txt")
# source_rows_path = Path("/content/drive/MyDrive/yhd_project/llm-main/gpt5.jsonl")

# VAL_RATIO = 0.1
# SEED = 42
# REQUIRED_KEYS = ("coherence", "consistency", "fluency", "relevance")
# def build_input(base_prompt: str, article: str, summary: str) -> str:
#     return (
#         f"{base_prompt}\n\n"
#         "This is the complete source article for which you will be evaluating:\n"
#         f"{article}\n\n"
#         "This is the LLM generated summary you are about to evaluate:\n"
#         f"{summary}\n\n"
#         "Return ONLY the final JSON object."
#     )


# def load_rows(path: Path) -> list[dict]:
#     if path.suffix.lower() == ".jsonl":
#         rows = []
#         with path.open("r", encoding="utf-8") as fin:
#             for line in fin:
#                 line = line.strip()
#                 if line:
#                     rows.append(json.loads(line))
#         return rows
#     return json.loads(path.read_text(encoding="utf-8"))


# def extract_json(raw: str) -> dict | None:
#     raw = (raw or "").strip()
#     if not raw:
#         return None
#     try:
#         return json.loads(raw)
#     except json.JSONDecodeError:
#         pass

#     start = raw.find("{")
#     end = raw.rfind("}")
#     if start == -1 or end == -1 or end <= start:
#         return None
#     candidate = raw[start : end + 1]
#     try:
#         return json.loads(candidate)
#     except json.JSONDecodeError:
#         return None


# def get_prompt_for_row(row: dict, base_prompt: str) -> str:
#     if isinstance(row.get("messages"), list) and row["messages"]:
#         for m in row["messages"]:
#             if m.get("role") == "user":
#                 return m.get("content", "")

#     article = row.get("original", "")
#     summary = row.get("summary", "")
#     if article and summary:
#         return build_input(base_prompt, article, summary)

#     raise ValueError("Row missing both `messages` and `original`+`summary`.")


# def extract_article_summary_from_prompt(prompt: str) -> tuple[str | None, str | None]:
#     a_marker = "This is the complete source article for which you will be evaluating:\n"
#     s_marker = "\n\nThis is the LLM generated summary you are about to evaluate:\n"
#     end_marker = "\n\nReturn ONLY the final JSON object."
#     a_start = prompt.find(a_marker)
#     s_start = prompt.find(s_marker)
#     if a_start == -1 or s_start == -1:
#         return None, None
#     article = prompt[a_start + len(a_marker) : s_start]
#     rem = prompt[s_start + len(s_marker) :]
#     end_pos = rem.rfind(end_marker)
#     summary = rem[:end_pos] if end_pos != -1 else rem
#     return article.strip() or None, summary.strip() or None



# base_prompt = PROMPT_PATH.read_text(encoding="utf-8")

# input_path = Path(DEFAULT_INPUT_PATH)
# output_path = Path(DEFAULT_OUTPUT_PATH)
# adapter_path = Path(DEFAULT_ADAPTER_PATH)
# output_path.parent.mkdir(parents=True, exist_ok=True)

# rows = load_rows(input_path)
# source_map: dict[str, dict] = {}
# source_rows_path = Path(source_rows_path)
# if source_rows_path.exists():
#     source_rows = load_rows(source_rows_path)
#     source_map = {str(r.get("id")): r for r in source_rows if r.get("id")}
# if LIMIT > 0:
#     rows = rows[:LIMIT]

# dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
# tokenizer = AutoTokenizer.from_pretrained(DEFAULT_BASE_MODEL, use_fast=True)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# base_model = AutoModelForCausalLM.from_pretrained(
#     DEFAULT_BASE_MODEL,
#     dtype=dtype,
#     device_map="auto" if torch.cuda.is_available() else None,
# )
# model = PeftModel.from_pretrained(base_model, DEFAULT_ADAPTER_PATH)
# model.eval()

# total = len(rows)
# with output_path.open("w", encoding="utf-8") as fout:
#     for i, row in enumerate(rows, start=1):
#         prompt = get_prompt_for_row(row, base_prompt)
#         messages = [{"role": "user", "content": prompt}]
#         inputs = tokenizer.apply_chat_template(
#             messages,
#             add_generation_prompt=True,
#             tokenize=True,
#             return_tensors="pt",
#             return_dict=True,
#         ).to(model.device)

#         with torch.no_grad():
#             outputs = model.generate(
#                 **inputs,
#                 max_new_tokens=max_new_tokens,
#                 do_sample=False,
#                 pad_token_id=tokenizer.eos_token_id,
#             )

#         new_tokens = outputs[0][inputs["input_ids"].shape[-1] :]
#         pred_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
#         pred_json = extract_json(pred_text)

#         out = {
#             "idx": i - 1,
#             "id": row.get("id", f"row_{i-1}"),
#             "dataset": row.get("dataset"),
#             "source_index": row.get("source_index"),
#             "prediction_text": pred_text,
#             "prediction_json": pred_json,
#             "parse_ok": pred_json is not None,
#         }
#         rec_id = str(out["id"])
#         source_row = source_map.get(rec_id)
#         if source_row:
#             out["dataset"] = out["dataset"] or source_row.get("dataset")
#             out["source_index"] = out["source_index"] or source_row.get("source_index")
#             out["original"] = source_row.get("original")
#             out["summary"] = source_row.get("summary")
#         else:
#             article, summary = extract_article_summary_from_prompt(prompt)
#             out["original"] = row.get("original") or article
#             out["summary"] = row.get("summary") or summary

#         if isinstance(row.get("messages"), list):
#             for m in row["messages"]:
#                 if m.get("role") == "assistant":
#                     out["reference_text"] = m.get("content", "")
#                     break
#         fout.write(json.dumps(out, ensure_ascii=False) + "\n")

#         if i % 25 == 0 or i == total:
#             print(f"[{i}/{total}] scored")

# print(f"Done. Wrote predictions to: {output_path}")



In [ ]:
# !pip install -q pandas scipy numpy

In [ ]:
# import json
# from pathlib import Path
# import pandas as pd
# import numpy as np
# from scipy.stats import spearmanr, pearsonr, kendalltau

# PRED_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/sft_val_qwen25_1.5b_lora_test_preds_gpt5.jsonl")
# GOLD_PATH = Path("/content/drive/MyDrive/yhd_project/llm-main/gpt5.jsonl")

# DIMENSIONS = ["coherence", "consistency", "fluency", "relevance"]


# def load_jsonl(path):
#     rows = []
#     with open(path, "r", encoding="utf-8") as f:
#         for line in f:
#             line = line.strip()
#             if line:
#                 rows.append(json.loads(line))
#     return rows


# def try_parse_json(x):
#     if x is None:
#         return None
#     if isinstance(x, dict):
#         return x
#     if isinstance(x, str):
#         x = x.strip()
#         if not x:
#             return None
#         try:
#             return json.loads(x)
#         except:
#             start = x.find("{")
#             end = x.rfind("}")
#             if start != -1 and end != -1 and end > start:
#                 try:
#                     return json.loads(x[start:end+1])
#                 except:
#                     return None
#     return None


# def extract_scores(obj):
#     obj = try_parse_json(obj)
#     out = {}
#     for dim in DIMENSIONS:
#         score = None
#         if isinstance(obj, dict) and isinstance(obj.get(dim), dict):
#             score = obj[dim].get("score")
#         out[dim] = score
#     return out

In [ ]:
# pred_rows = load_jsonl(PRED_PATH)
# gold_rows = load_jsonl(GOLD_PATH)

# pred_data = []
# for row in pred_rows:
#     pred_json = row.get("prediction_json")
#     scores = extract_scores(pred_json)
#     rec = {"id": str(row.get("id"))}
#     for dim in DIMENSIONS:
#         rec[f"pred_{dim}"] = scores[dim]
#     pred_data.append(rec)

# gold_data = []
# for row in gold_rows:
#     gold_json = row.get("raw_response_text")
#     scores = extract_scores(gold_json)
#     rec = {"id": str(row.get("id"))}
#     for dim in DIMENSIONS:
#         rec[f"gold_{dim}"] = scores[dim]
#     gold_data.append(rec)

# pred_df = pd.DataFrame(pred_data)
# gold_df = pd.DataFrame(gold_data)

# merged = pred_df.merge(gold_df, on="id", how="inner")

# for dim in DIMENSIONS:
#     merged[f"pred_{dim}"] = pd.to_numeric(merged[f"pred_{dim}"], errors="coerce")
#     merged[f"gold_{dim}"] = pd.to_numeric(merged[f"gold_{dim}"], errors="coerce")

# print("pred rows:", len(pred_df))
# print("gold rows:", len(gold_df))
# print("matched rows:", len(merged))

# merged.head()

In [ ]:
# def safe_corr(fn, x, y):
#     mask = ~(pd.isna(x) | pd.isna(y))
#     x = np.array(x[mask], dtype=float)
#     y = np.array(y[mask], dtype=float)
#     if len(x) < 2:
#         return np.nan
#     try:
#         val = fn(x, y)[0]
#         return val
#     except:
#         return np.nan


# def evaluate_dimension(df, dim):
#     pred = df[f"pred_{dim}"]
#     gold = df[f"gold_{dim}"]

#     mask = ~(pd.isna(pred) | pd.isna(gold))
#     pred = np.array(pred[mask], dtype=float)
#     gold = np.array(gold[mask], dtype=float)

#     if len(pred) == 0:
#         return {
#             "dimension": dim,
#             "n": 0,
#             "spearman": np.nan,
#             "pearson": np.nan,
#             "kendall": np.nan,
#             "mae": np.nan,
#             "rmse": np.nan,
#             "exact_match": np.nan,
#             "off_by_1_acc": np.nan,
#             "pred_mean": np.nan,
#             "gold_mean": np.nan,
#         }

#     return {
#         "dimension": dim,
#         "n": len(pred),
#         "spearman": safe_corr(spearmanr, pred, gold),
#         "pearson": safe_corr(pearsonr, pred, gold),
#         "kendall": safe_corr(kendalltau, pred, gold),
#         "mae": np.mean(np.abs(pred - gold)),
#         "rmse": np.sqrt(np.mean((pred - gold) ** 2)),
#         "exact_match": np.mean(pred == gold),
#         "off_by_1_acc": np.mean(np.abs(pred - gold) <= 1),
#         "pred_mean": np.mean(pred),
#         "gold_mean": np.mean(gold),
#     }


# eval_rows = [evaluate_dimension(merged, dim) for dim in DIMENSIONS]
# eval_df = pd.DataFrame(eval_rows)

# eval_df

In [ ]:
# eval_df_rounded = eval_df.copy()

# for col in ["spearman", "pearson", "kendall", "mae", "rmse", "exact_match", "off_by_1_acc", "pred_mean", "gold_mean"]:
#     eval_df_rounded[col] = eval_df_rounded[col].round(4)

# eval_df_rounded

In [ ]:
# OUT_DIR = Path("/content/drive/MyDrive/yhd_project/llm-main/eval_tables_gpt5")
# OUT_DIR.mkdir(parents=True, exist_ok=True)

# eval_df_rounded.to_csv(OUT_DIR / "eval_tables_gpt5.csv", index=False)

